# Demo Recommender — CNTT-KLCN257 (CBF TF-IDF + CF SVD + Hybrid fusion)

Muc tieu de cuong: Content-Based (TF-IDF cosine) + Collaborative (MF-SVD) + Hybrid co trong so hoc + MCDM/WSM + LLM/RAG.
- WSM cu duoc GIU NGUYEN (`app/core/wsm_engine.py`, endpoint `/api/v1/ai/wsm/rank`).
- Hybrid them endpoint `/api/v1/ai/hybrid/rank` (Backend `suggest-top3-hybrid` fallback ve cu).
- Trong so alpha/beta/gamma HOC tu du lieu (`notebooks/learn_hybrid_weights.py`), khong mac dinh.
- Du lieu hien tai: SYNTHETIC `data/wedding_dataset_5k.csv`; data that thu qua `app/core/export_reviews.py` -> `data/wedding_reviews_real.csv`.

In [ ]:
import json, sys
from pathlib import Path
BASE = Path.cwd()
if (BASE / 'app').exists():
    pass
elif (BASE / 'AI_Service').exists():  # chay tu repo root
    BASE = BASE / 'AI_Service'
sys.path.insert(0, str(BASE))
print('BASE =', BASE)

In [ ]:
# 1) CBF: TF-IDF tren menu/decor/hall (Aggarwal 2016 Ch.3; Ricci Handbook; sklearn TfidfVectorizer)
from app.core import cbf_engine
cat = json.loads((BASE / 'data' / 'raw_catalog.json').read_text(encoding='utf-8'))
hall, menu, decor = cat['halls'][0], cat['menus'][0], cat['decors'][0]
print('CBF lang-man vs Amur/TD001/Endless Love =', round(cbf_engine.cbf_score('Lãng mạn', [], menu, decor, hall), 4))
print('CBF hien-dai vs cung combo =', round(cbf_engine.cbf_score('Hiện đại', [], menu, decor, hall), 4))

In [ ]:
# 2) CF: user-item (SampleId x Hall_Menu_Decor, rating=Satisfaction) + TruncatedSVD n=20 (Koren et al. 2009)
from app.core import cf_engine
st = cf_engine.fit()
print(f"users={len(st['users'])} items={len(st['items'])} global_mean={st['global_mean']:.3f}")
cat = json.loads((BASE / 'data' / 'raw_catalog.json').read_text(encoding='utf-8'))
h, m, d = cat['halls'][3], cat['menus'][21], cat['decors'][3]
print('CF cold-start s_cf =', round(cf_engine.cf_score(None, h, m, d), 4))
print('CF user=1 s_cf =', round(cf_engine.cf_score(1, h, m, d), 4))

In [ ]:
# 3) Hybrid fusion: s_final = alpha*s_cf + beta*s_cbf + gamma*S_wsm (hoc duoc, khong mac dinh 0.2/0.2/0.6)
from app.core import hybrid_engine, wsm_engine
w = hybrid_engine.load_hybrid_weights()
print('Learned weights:', {k: round(float(v), 4) for k, v in w.items() if not k.startswith('_')})
s_cf, s_cbf, s_wsm = 0.81, 0.16, 0.72
print('fuse demo =', round(hybrid_engine.fuse(s_cf, s_cbf, s_wsm, w), 4))

In [ ]:
# 4) End-to-end: Hybrid rank 1 request (hard-filter nhu WSM + diversity Top3 tai su dung)
from datetime import date, timedelta
from app.api.v1.endpoints.hybrid import HybridRankRequest, rank_hybrid
d = (date.today() + timedelta(days=60)).isoformat()
req = HybridRankRequest(guest_count=250, table_count=25, expected_budget=300_000_000, event_date=d, shift='Ca trua', desired_style='Lãng mạn', priority_services=[], occupied_hall_ids=[])
res = rank_hybrid(req)
print('total =', res.total_candidates_evaluated, '| top =', len(res.top_3))
def _g(t, k):
    return t[k] if isinstance(t, dict) else getattr(t, k)
for t in res.top_3:
    print(_g(t, 'hall_name'), '|', _g(t, 'menu_name'), '|', _g(t, 'decor_name'), '->', _g(t, 'matching_score'))

## Ghi chu thuc nghiem & du lieu

- Chay `python notebooks/learn_hybrid_weights.py` de hoc (alpha, beta, gamma) — log ra console + `app/core/hybrid_weights.json`.
- Chay `python notebooks/evaluate_hybrid.py` de so P@3/R@3/NDCG@3/HitRate@3 (WSM/CBF/CF/Hybrid, cung 1000 test x 20 candidates) -> `output_plots/hybrid_comparison.png`.
- SYNTHETIC vs THAT: hien tai chi co synthetic; khi `data/wedding_reviews_real.csv` du >=20 dong that thi script tu chay cot REAL rieng, khong tron lan.